# AEGIS — smoke test + train nudity (Colab A100)

Chạy tuần tự từng cell. Trước khi bắt đầu:

1. **Runtime → Change runtime type → A100 GPU** rồi Save. (Bật runtime là tính units.)
2. Chuẩn bị **HF token** đã accept license `CompVis/stable-diffusion-v-1-4-original` (https://huggingface.co/settings/tokens).
3. Giữ tab này mở suốt lúc train (đóng tab → runtime ngắt).

Ngân sách: setup env ~20', tải ckpt ~3', smoke ~3', train nudity ~2h → tổng ~2.5h A100 (còn dư trong 33 units ≈ ~7h).

Kết quả cuối: `Diffusers-UNet-full-nudity-epoch_999.pt` (3.4 GB) — file này là artifact duy nhất để chạy eval/attack sau này.

## 1. Kiểm tra GPU (phải là A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Clone repo

In [ ]:
import os
if not os.path.isdir('/content/security-project'):
    !git clone https://github.com/izahai/security-project.git /content/security-project
%cd /content/security-project

## 3. Miniconda + tạo env `AEGIS` (~20 phút)

Env pin py3.8.5 / torch1.11 / cuda11.3 (torch 1.11 cu113 hỗ trợ A100 sm_80). Colab không có conda sẵn nên cài miniconda vào `/opt/conda`. Idempotent — chạy lại không tạo trùng.

In [ ]:
%%bash
set -e
if [ ! -x /opt/conda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-2-Linux-x86_64.sh -O /tmp/mc.sh
  bash /tmp/mc.sh -b -p /opt/conda
fi
export PATH=/opt/conda/bin:$PATH
conda config --set channel_priority flexible
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
if conda env list | grep -qE '^AEGIS[[:space:]]'; then
  echo 'env AEGIS da ton tai, bo qua'
else
  cd /content/security-project
  conda env create -f environment.yaml
fi
/opt/conda/bin/conda run -n AEGIS python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 4. HF token + tải SD v1.4 ckpt (7.7 GB)

Token nhập bằng `getpass` nên không hiện ra output/notebook.

In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('HF token (da accept license CompVis SD-v1-4): ')

In [ ]:
%%bash
cd /content/security-project
mkdir -p models
if [ ! -f models/sd-v1-4-full-ema.ckpt ]; then
  curl -L -H "Authorization: Bearer $HF_TOKEN" -o models/sd-v1-4-full-ema.ckpt \
    https://huggingface.co/CompVis/stable-diffusion-v-1-4-original/resolve/main/sd-v1-4-full-ema.ckpt
fi
ls -lh models/sd-v1-4-full-ema.ckpt

## 5. Smoke test — 2 iterations

Chỉ để chứng minh pipeline chạy được và sinh **cả hai** file `.pt`. `print(w)` mỗi step là log bình thường của repo (không phải lỗi).

In [ ]:
%%bash
cd /content/security-project
/opt/conda/bin/conda run -n AEGIS --no-capture-output python train-scripts/AEGIS.py \
  --prompt nudity --train_method full --attack_init random \
  --iterations 2 --save_interval 2
echo '--- output files ---'
ls -lh results/results_with_AEGIS/AEGIS/models/

### 5b. Verify — Diffusers ckpt load được vào UNet2DConditionModel (strict)

Đây là mắt nối duy nhất giữa repo train và repo attack/eval. Phải pass `strict=True`.

In [ ]:
%%bash
cd /content/security-project
/opt/conda/bin/conda run -n AEGIS --no-capture-output python - <<'PY'
import os, torch
from diffusers import UNet2DConditionModel
p = 'results/results_with_AEGIS/AEGIS/models/Diffusers-UNet-full-nudity-epoch_1.pt'
sd = torch.load(p, map_location='cpu')
unet = UNet2DConditionModel.from_pretrained(
    'CompVis/stable-diffusion-v1-4', subfolder='unet',
    use_auth_token=os.environ.get('HF_TOKEN'))
unet.load_state_dict(sd)  # strict=True mac dinh
print('OK: strict load thanh cong,', len(sd), 'tensors')
PY

## 6. Train nudity thật — 1000 iterations (~2h)

`--save_interval 1000` → chỉ lưu checkpoint cuối (`epoch_999`), tránh ghi 5 file thừa. Peak VRAM ~26 GB, A100 40 GB thoải mái. Cell này chạy ~2h, để tab mở.

In [ ]:
%%bash
cd /content/security-project
/opt/conda/bin/conda run -n AEGIS --no-capture-output python train-scripts/AEGIS.py \
  --prompt nudity --train_method full --attack_init random \
  --iterations 1000 --save_interval 1000
echo '--- output files ---'
ls -lh results/results_with_AEGIS/AEGIS/models/

## 7. Lưu checkpoint ra Google Drive

Chỉ cần file `Diffusers-*.pt` (3.4 GB) cho eval/attack về sau. Copy sang Drive để tải về sau (bền hơn `files.download` với file lớn).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
src = 'results/results_with_AEGIS/AEGIS/models/Diffusers-UNet-full-nudity-epoch_999.pt'
dst = '/content/drive/MyDrive/aegis/Diffusers-UNet-full-nudity-epoch_999.pt'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(src, dst)
print('da copy ->', dst, os.path.getsize(dst) / 1e9, 'GB')